In [12]:
#organiser et fusionner des fichiers CSV de données financières
import pandas as pd
import os
from glob import glob

# --- 1. Définir les dossiers où sont les fichiers CSV ---
prices_folder = "data/prices"          
fundamentals_folder = "data/fundamentals"  

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files = glob(os.path.join(prices_folder, "*.csv"))

prices_list = []
for file in price_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list.append(df)

prices_df = pd.concat(prices_list, ignore_index=True)
print("Prices dataset shape:", prices_df.shape)

# --- 3. Lire et concaténer tous les fichiers fundamentals ---
fund_files = glob(os.path.join(fundamentals_folder, "*.csv"))

funds_list = []
for file in fund_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2], low_memory=False)  # skip Ticker et ligne vide
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split("_")[0]  # ex: 'AAPL_fundamentals.csv'
    df["Ticker"] = ticker
    
    funds_list.append(df)

fundamentals_df = pd.concat(funds_list, ignore_index=True)
print("Fundamentals dataset shape:", fundamentals_df.shape)

# Pour prices_df
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]

# Pour fundamentals_df
cols = ["Ticker"] + [c for c in fundamentals_df.columns if c != "Ticker"]
fundamentals_df = fundamentals_df[cols]

print(prices_df.head())

print(fundamentals_df.head())

# --- 4. Sauvegarder les datasets fusionnés ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


Prices dataset shape: (2776288, 8)
Fundamentals dataset shape: (2439, 334)
  Ticker       Date  Adj Close      Close       High        Low       Open  \
0      A 2000-01-03  43.113316  51.502148  56.464592  48.193848  56.330471   
1      A 2000-01-04  39.819954  47.567955  49.266811  46.316166  48.730328   
2      A 2000-01-05  37.349922  44.617310  47.567955  43.141991  47.389126   
3      A 2000-01-06  35.927761  42.918453  44.349072  41.577251   44.08083   
4      A 2000-01-07  38.921741  46.494991  47.165592  42.203148  42.247852   

      Volume  
0  4674353.0  
1  4765083.0  
2  5758642.0  
3  2534434.0  
4  2819626.0  
  Ticker       Date  Tax Effect Of Unusual Items  Tax Rate For Calcs  \
0   AAPL 2024-09-30                          0.0            0.210000   
1   AAPL 2024-12-31                          0.0            0.147000   
2   AAPL 2025-03-31                          0.0            0.155000   
3   AAPL 2025-06-30                          0.0            0.164000   
4   AA

In [13]:
#nettoyer les données:
# Supprimer la colonne Close puisque Close_Adj est plus pertinente ( elle tient compte des dividendes et fractionnements d'actions)
if "Close" in prices_df.columns:
    prices_df.drop(columns=["Close"], inplace=True)

# Réordonner les colonnes pour mettre Ticker en premier
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]
print(prices_df.head())


# --- Filtrer les données à partir de 2024 pour travailler sur des données recentes ---
prices_df = prices_df[prices_df["Date"] >= "2024-01-01"]
print("Filtered Prices dataset shape:", prices_df.shape)


#nettoyer fundamentals_df en gardant uniquement les colonnes essentielles pour calculer des ratios financiers
# Colonnes essentielles pour les ratios fondamentaux
cols_to_keep = [
    # Identifiants
    "Ticker", "Date",
    
    # Revenus et bénéfices
    "Total Revenue", "Operating Revenue", "EBITDA", "EBIT", "Operating Income",
    "Net Income", "Net Income From Continuing Operations", "Net Income Common Stockholders",
    
    # Actions et EPS
    "Diluted Average Shares", "Basic Average Shares", "Diluted EPS", "Basic EPS",
    
    # Bilan
    "Total Debt", "Net Debt", "Current Assets", "Current Liabilities", "Cash Cash Equivalents And Short Term Investments",
    "Accounts Receivable", "Inventory", "Invested Capital", "Total Equity", "Common Stock Equity",
    
    # Coût
    "Cost Of Revenue"
]
# Sélectionner uniquement les colonnes qui existent dans le DataFrame fundamentals_df
cols_to_keep_existing = [col for col in cols_to_keep if col in fundamentals_df.columns]

# Conserver uniquement les colonnes nécessaires
fundamentals_df_clean = fundamentals_df[cols_to_keep_existing]

fundamentals_df=fundamentals_df_clean

print("Cleaned Fundamentals dataset shape:", fundamentals_df.shape)

# --- 4. Sauvegarder  ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


  Ticker       Date  Adj Close       High        Low       Open     Volume
0      A 2000-01-03  43.113316  56.464592  48.193848  56.330471  4674353.0
1      A 2000-01-04  39.819954  49.266811  46.316166  48.730328  4765083.0
2      A 2000-01-05  37.349922  47.567955  43.141991  47.389126  5758642.0
3      A 2000-01-06  35.927761  44.349072  41.577251   44.08083  2534434.0
4      A 2000-01-07  38.921741  47.165592  42.203148  42.247852  2819626.0
Filtered Prices dataset shape: (126639, 7)
Cleaned Fundamentals dataset shape: (2439, 24)


In [14]:
# lire les données dans prices_2025 pour recuperer les données de 2025 ( prices_df contient actuellement les données de 2024 seulement )

prices_folder_2025 = "data/prices_2025"          

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files_2025 = glob(os.path.join(prices_folder_2025, "*.csv"))

prices_list_2025 = []#une liste de dataframes
for file in price_files_2025:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list_2025.append(df)

prices_df_2025 = pd.concat(prices_list_2025, ignore_index=True)
print("Prices dataset shape:", prices_df_2025.shape)

# 1. Supprimer la colonne 'Close' de prices_df_2025 si elle existe
if 'Close' in prices_df_2025.columns:
    prices_df_2025 = prices_df_2025.drop(columns=['Close'])

# 2. Filtrer pour ne garder que les dates >= 2024
prices_df_2025['Date'] = pd.to_datetime(prices_df_2025['Date'])
prices_df_2025 = prices_df_2025[prices_df_2025['Date'].dt.year >= 2024]

# 3. Concaténer avec prices_df
combined_prices = pd.concat([prices_df, prices_df_2025], ignore_index=True)

# 4. Supprimer les doublons basés sur Ticker + Date
combined_prices = combined_prices.drop_duplicates(subset=['Ticker', 'Date'], keep='first')

# 5. Trier par Ticker puis Date
combined_prices = combined_prices.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# 6. Mettre à jour prices_df
prices_df = combined_prices

print("Updated prices_df shape:", prices_df.shape)

#enregistrer le nouveau fichier CSV avec les données mises à jour
prices_df.to_csv("all_prices.csv", index=False)

#en effet maintenant chaque jour dans prices ( 2024 ou 2025) peut trouver le rapport fondamental le plus récent dans fundamentals_df (exemple: pour un rapport ds fundamentals mis dans decembre, il sera utilisé pour tous les jours de dec, jan et fev jusqu'au prochain rapport fondamental en mars)

Prices dataset shape: (2891473, 8)
Updated prices_df shape: (241824, 7)


In [ ]:
#ajouter des ratios financiers dans fundamentals_df
# --- 1. Vérifier que les colonnes nécessaires sont présentes
required_columns = [
    'Ticker', 'Date', 'Total Revenue', 'Operating Revenue', 'EBITDA', 'EBIT',
    'Operating Income', 'Net Income', 'Net Income From Continuing Operations',
    'Net Income Common Stockholders', 'Diluted Average Shares', 'Basic Average Shares',
    'Diluted EPS', 'Basic EPS', 'Total Debt', 'Net Debt', 'Current Assets',
    'Current Liabilities', 'Cash Cash Equivalents And Short Term Investments',
    'Accounts Receivable', 'Inventory', 'Invested Capital', 'Common Stock Equity',
    'Cost Of Revenue'
]

missing_cols = [c for c in required_columns if c not in fundamentals_df.columns]
if missing_cols:
    print("Warning: Missing columns:", missing_cols)

# --- 2. Calculer les ratios directement dans fundamentals_df
fundamentals_df['ROE'] = fundamentals_df['Net Income Common Stockholders'] / fundamentals_df['Common Stock Equity']
fundamentals_df['Profit_Margin'] = fundamentals_df['Net Income'] / fundamentals_df['Total Revenue']
fundamentals_df['Debt_to_Equity'] = fundamentals_df['Total Debt'] / fundamentals_df['Common Stock Equity']

# EPS Growth par ticker
fundamentals_df.sort_values(by=['Ticker', 'Date'], inplace=True)
fundamentals_df['EPS_Growth'] = fundamentals_df.groupby('Ticker')['Diluted EPS'].pct_change(fill_method=None)

fundamentals_df['Current_Ratio'] = fundamentals_df['Current Assets'] / fundamentals_df['Current Liabilities']
fundamentals_df['Quick_Ratio'] = (fundamentals_df['Current Assets'] - fundamentals_df['Inventory']) / fundamentals_df['Current Liabilities']
fundamentals_df['Operating_Margin'] = fundamentals_df['Operating Income'] / fundamentals_df['Total Revenue']

# --- 3. Ajouter le mois de fin d'année fiscale (puisque j'ai pas de colonnes specifiques )  cad  le dernier rapport est en decembre ( 12 est divisible par 3 = trimestre )
fundamentals_df['Fiscal_Year_End_Month'] = 12

# --- 4. Vérifier le résultat
print(fundamentals_df[['Ticker', 'Date', 'ROE', 'Profit_Margin', 'Debt_to_Equity',
                       'EPS_Growth', 'Current_Ratio', 'Quick_Ratio', 'Operating_Margin']].head())

# --- 5. Sauvegarder les données mises à jour ---
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


    Ticker       Date       ROE  Profit_Margin  Debt_to_Equity  EPS_Growth  \
234      A 2024-10-31  0.059512       0.206349        0.574771         NaN   
235      A 2025-01-31  0.052763       0.189173        0.557989   -0.090164   
236      A 2025-04-30  0.035039       0.128897        0.569589   -0.324324   
237      A 2025-07-31  0.052747       0.193326        0.535165    0.573333   
238      A 2025-10-31       NaN            NaN             NaN    0.296610   

     Current_Ratio  Quick_Ratio  Operating_Margin  
234       2.089182     1.576253          0.239859  
235       2.197432     1.663991          0.223676  
236       2.091451     1.598907          0.179856  
237       2.247886     1.711945          0.207135  
238            NaN          NaN               NaN  
